In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Hub Identification and Pathway Enrichment Analysis
===================================================
This script builds a directed graph from an XML file containing pathway annotations
(e.g., generated by 02_network_construction_llm_annotation.py), computes node
centralities (PageRank and Degree), performs community detection using the Louvain
algorithm, and conducts Reactome and KEGG enrichment analyses on gene sets derived
from the network.

Dependencies:
    - networkx, pandas, numpy, matplotlib, seaborn
    - gseapy, mygene, python-louvain
    - scipy

Usage:
    python 03_hub_identification_pathway_enrichment.py
"""

import os
import xml.etree.ElementTree as ET
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mygene
import gseapy as gp
import community as community_louvain
import warnings

warnings.filterwarnings('ignore')

# ----------------------------------------------------------------------
# 1. Build graph from XML
# ----------------------------------------------------------------------
def clean_gene_name(name):
    """Extract a plausible gene symbol from a node name."""
    name = name.rstrip('*')
    if '+' in name:
        name = name.split('+')[0]
    name = name.replace('(cyto)', '').replace('(mito)', '').replace('(outer)', '').replace('(inner)', '').strip()
    return name

def build_graph_from_xml(xml_path):
    """Parse XML and return a directed graph with node attributes."""
    if not os.path.exists(xml_path):
        raise FileNotFoundError(f"XML file not found: {xml_path}")

    tree = ET.parse(xml_path)
    root = tree.getroot()
    G = nx.DiGraph()

    for pathway in root.findall('pathway'):
        node_list = pathway.find('nodeList')
        if node_list is not None:
            for node in node_list.findall('node'):
                node_name = node.get('name')
                if node_name:
                    cleaned = clean_gene_name(node_name)
                    G.add_node(node_name, cleaned=cleaned)

        edge_list = pathway.find('edgeList')
        if edge_list is not None:
            for edge in edge_list.findall('edge'):
                start = edge.find('startNode')
                end = edge.find('endNode')
                if start is not None and end is not None:
                    start_node = start.get('node')
                    end_node = end.get('node')
                    if start_node and end_node:
                        G.add_edge(start_node, end_node)

    print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    return G

# ----------------------------------------------------------------------
# 2. Centrality analysis
# ----------------------------------------------------------------------
def compute_centralities(G):
    """Compute PageRank and degree centrality."""
    deg_cent = nx.degree_centrality(G)
    pr = nx.pagerank(G, alpha=0.85)

    nodes = list(G.nodes())
    df_cent = pd.DataFrame({
        'Node': nodes,
        'Degree_Centrality': [deg_cent.get(n, 0) for n in nodes],
        'PageRank': [pr.get(n, 0) for n in nodes],
    })
    df_cent = df_cent.sort_values('PageRank', ascending=False)
    return df_cent

def plot_pagerank_top10(df_cent, output_file='pagerank_top10.png'):
    """Bar plot of top 10 PageRank nodes."""
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_cent.head(10), x='PageRank', y='Node', palette='viridis')
    plt.title('Top 10 Key Proteins by PageRank')
    plt.tight_layout()
    plt.savefig(output_file, dpi=300)
    plt.show()

# ----------------------------------------------------------------------
# 3. Community detection and Reactome enrichment
# ----------------------------------------------------------------------
def is_gene_candidate(name):
    """Filter out non‑gene nodes (e.g., ions, small molecules)."""
    if not name:
        return False
    blacklist = [' ', '/', '(', ')', '-', 'Ca2', 'cAMP', 'ROS', 'QH2', 'H2O', 'PIP3', 'IP3', 'DAG']
    if any(b in name for b in blacklist):
        return False
    # Accept names containing only letters and digits, length > 1
    if all(c.isalpha() or c.isdigit() for c in name) and len(name) > 1:
        return True
    return False

def community_enrichment(G, partition):
    """For each community, extract gene symbols and perform Reactome enrichment."""
    n_communities = len(set(partition.values()))
    for comm_id in range(n_communities):
        comm_nodes = [node for node, c in partition.items() if c == comm_id]
        gene_symbols = []
        for n in comm_nodes:
            cleaned = G.nodes[n].get('cleaned', n)
            if is_gene_candidate(cleaned):
                gene_symbols.append(cleaned)
        gene_symbols = list(set(gene_symbols))
        if len(gene_symbols) < 5:
            continue

        print(f"\n--- Community {comm_id} (nodes: {len(comm_nodes)}, candidate genes: {len(gene_symbols)}) ---")
        try:
            enr = gp.enrichr(gene_list=gene_symbols,
                             gene_sets='Reactome_2022',
                             organism='human',
                             cutoff=0.05,
                             verbose=False)
            if not enr.results.empty:
                top3 = enr.results[['Term', 'Adjusted P-value']].head(3)
                print("Top 3 Reactome pathways:")
                print(top3.to_string(index=False))
            else:
                print("   No significant Reactome pathways (p < 0.05)")
        except Exception as e:
            print(f"   Enrichment failed: {e}")

def plot_community_structure(G_und, partition, output_file='community_structure.png'):
    """Plot the largest connected component colored by community."""
    # Get largest connected component
    largest_cc = max(nx.connected_components(G_und), key=len)
    G_largest = G_und.subgraph(largest_cc)
    pos = nx.spring_layout(G_largest, k=0.3, iterations=50)
    colors = [partition[node] for node in G_largest.nodes()]

    plt.figure(figsize=(12, 10))
    nx.draw_networkx_nodes(G_largest, pos, node_color=colors, cmap=plt.cm.jet, node_size=30)
    nx.draw_networkx_edges(G_largest, pos, alpha=0.2, edge_color='gray')
    plt.title("Community Structure of the Largest Connected Component")
    plt.axis('off')
    plt.savefig(output_file, dpi=300)
    plt.show()

# ----------------------------------------------------------------------
# 4. KEGG enrichment for all candidate genes and for top PageRank nodes
# ----------------------------------------------------------------------
def kegg_enrichment_all_genes(G):
    """KEGG enrichment for all gene candidates in the graph."""
    gene_names = list(G.nodes())
    cleaned_names = [G.nodes[n].get('cleaned', n) for n in gene_names]
    gene_candidates = [name for name in cleaned_names if is_gene_candidate(name)]
    print(f"Total candidate genes: {len(gene_candidates)}")

    mg = mygene.MyGeneInfo()
    results = mg.querymany(gene_candidates, scopes='symbol', fields='entrezgene',
                           species='human', returnall=True, verbose=False)
    entrez_ids = []
    for item in results['out']:
        if 'entrezgene' in item and item['entrezgene'] is not None:
            entrez = item['entrezgene']
            if isinstance(entrez, list):
                entrez = entrez[0]
            entrez_ids.append(str(entrez))
    print(f"Successfully mapped: {len(entrez_ids)} / {len(gene_candidates)}")

    if len(entrez_ids) == 0:
        print("No genes mapped. Skipping enrichment.")
        return

    enr = gp.enrichr(gene_list=entrez_ids,
                     gene_sets='KEGG_2021_Human',
                     organism='human',
                     cutoff=0.05)
    if enr.results.empty:
        print("No significantly enriched KEGG pathways (p < 0.05)")
    else:
        print(f"Found {len(enr.results)} significantly enriched KEGG pathways")
        enr.results.to_csv('kegg_enrichment_all_genes.csv', index=False)

def kegg_enrichment_top200(G, df_cent):
    """KEGG enrichment for top 200 PageRank nodes."""
    top_nodes = df_cent.head(200)['Node'].tolist()
    cleaned_top = [clean_gene_name(g) for g in top_nodes]
    gene_symbols_top = [g for g in cleaned_top if is_gene_candidate(g)]
    print(f"Filtered {len(gene_symbols_top)} valid gene symbols from top 200 PageRank nodes")

    enr_top = gp.enrichr(gene_list=gene_symbols_top,
                         gene_sets='KEGG_2021_Human',
                         organism='human',
                         cutoff=0.05,
                         verbose=True)
    if not enr_top.results.empty:
        print(f"Found {len(enr_top.results)} significantly enriched KEGG pathways")
        enr_top.results.to_csv('kegg_enrichment_top200.csv', index=False)
        # Bubble plot for top 10
        top10 = enr_top.results.head(10)
        plt.figure(figsize=(12, 8))
        sns.scatterplot(data=top10, x='Adjusted P-value', y='Term',
                        size='Odds Ratio', sizes=(50, 300),
                        hue='Odds Ratio', palette='coolwarm')
        plt.xscale('log')
        plt.title('Top 10 Enriched KEGG Pathways (Top 200 PageRank nodes)')
        plt.tight_layout()
        plt.savefig('kegg_enrichment_top200_bubble.png', dpi=300)
        plt.show()
    else:
        print("No significant KEGG pathways (p < 0.05)")

# ----------------------------------------------------------------------
# 5. Main pipeline
# ----------------------------------------------------------------------
def main():
    # Input XML file (generated by previous step)
    xml_file = "oncogene_pathways.xml"

    # Build graph
    G = build_graph_from_xml(xml_file)

    # Centrality analysis
    print("\nComputing centralities...")
    df_cent = compute_centralities(G)

    print("\n=== Top 20 key proteins by PageRank ===")
    print(df_cent.head(20)[['Node', 'PageRank', 'Degree_Centrality']])

    plot_pagerank_top10(df_cent)
    df_cent.to_csv('key_proteins.csv', index=False)
    print("Key proteins saved to key_proteins.csv")

    # Community detection (Louvain)
    G_und = G.to_undirected()
    partition = community_louvain.best_partition(G_und, random_state=42)
    print(f"\nDetected {len(set(partition.values()))} communities")

    plot_community_structure(G_und, partition)

    # Reactome enrichment per community
    print("\n--- Reactome enrichment per community ---")
    community_enrichment(G, partition)

    # KEGG enrichment for all genes and for top PageRank genes
    print("\n--- KEGG enrichment for all candidate genes ---")
    kegg_enrichment_all_genes(G)

    print("\n--- KEGG enrichment for top 200 PageRank nodes ---")
    kegg_enrichment_top200(G, df_cent)

if __name__ == "__main__":
    main()